# 🚀 Fine-Tune Llama 3.2 3B for Lordy's Assistant
**Train a custom AI assistant for betting analysis, travel planning, and personal tasks**
Run this on **Google Colab's free T4 GPU** — Runtime → Change runtime type → T4 GPU

In [ ]:
# Cell 1: Install Unsloth (fast training library)
# Colab ships with transformers 5.5.0+ which breaks Unsloth - downgrade first
!pip install "transformers<5.5.0" --quiet
!pip install unsloth
!pip install --force-reinstall --no-deps unsloth xformers trl peft accelerate bitsandbytes
print("✅ Done installing")


In [ ]:
# Cell 2: Load Llama 3.2 3B in 4-bit
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Llama-3.2-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print('✅ Model loaded')

In [ ]:
# Cell 3: Add LoRA adapters (efficient fine-tuning)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print('✅ LoRA configured')

In [ ]:
# Cell 4: Your Personal Training Data
from datasets import Dataset

train_data = [
    {'instruction': 'Analyze betting odds for Argentina vs Egypt at 1.27',
     'output': 'Argentina at 1.27 gives ~70% win probability. World Champions with Messi, Alvarez, Martinez. $100 bet pays $127.'},
    {'instruction': 'What is the win rate for Morocco at odds 1.77?',
     'output': 'Morocco at 1.77 gives ~51% win probability. Below 70% threshold but favored. Canada weaker but still competitive.'},
    {'instruction': 'What are good bets on this screen? Argentina 1.27, Colombia 1.35, Brazil 1.63',
     'output': 'Argentina 1.27 (70% ✅) is safest. Colombia 1.35 (65%) is good value. Brazil 1.63 (54%) is risky - Norway has Haaland.'},
    {'instruction': 'Plan Singapore to Macau to cold China city in December',
     "output": "From Macau: Beijing (3hr flight, $80-150, -5°C snow), Xi'an (2.5hr, $50-100, 0-8°C snow possible), Harbin (4hr, $100-200, -20°C guaranteed). Best: Beijing."},
    {'instruction': 'What hotels near KSL Mall Johor Bahru?',
     'output': 'KSL Resort (4★, ~$66/night, connected to KSL Mall, pool, hot tub), Fives Hotel (3★, ~$38, 3min walk), Grand Paragon (4★, ~$35, 5min drive).'},
    {'instruction': 'What is my passive income from Nexo?',
     'output': '$93,000 USDT at 8.25% APY = ~$7,672/year or $639/month passive income. No lockup. Best rate in market.'},
    {'instruction': 'What train from Kansai Airport to Namba Osaka?',
     'output': 'Nankai Rapi:t limited express - 34-38 min, ~1,400 yen direct to Namba Station. Best option.'},
    {'instruction': 'What should I buy in Japan?',
     'output': 'Medicines: Pabron Gold A, Bufferin, Salonpas, Rohto eye drops. Skincare: Anessa sunscreen, Hada Labo, DHC Lip Cream. Snacks: unique Kit Kat flavors, matcha.'},
    {'instruction': 'Estimate USDT interest on $93,000 at 8.25% for 1 month',
     'output': '$93,000 × 8.25% ÷ 12 = ~$639/month or ~$21.30/day passive income. Taxable in Singapore.'},
    {'instruction': 'Compare DeepSeek V4 Flash vs Llama 3.2 3B',
     'output': 'DeepSeek V4 Flash is ~600B MoE model - much more powerful, costs $0.14/M tokens. Llama 3.2 3B is much smaller, runs locally for free, good for basic tasks and offline fallback.'},
    {'instruction': 'Is my Mac mini good for local AI?',
     "output": "Your Mac mini is Intel i5-8500B with 8GB RAM. Can run 1-3B models comfortably. 7B models are slow. 17B+ models won't fit. Best to use DeepSeek API for heavy work and Llama 3.2 3B for free offline tasks."},
    {'instruction': 'What should I do with my $97K in crypto?',
     'output': '$93K USDT earning 8.25% on Nexo ($639/mo) + $4K NEXO tokens. This is good passive income. Maintain as emergency fund if you plan to buy HDB. Consider moving some to Binance USD1 flex (10.5%) for better rate.'},
]

def fmt(entry):
    return f"### Instruction:\n{entry['instruction']}\n\n### Response:\n{entry['output']}{tokenizer.eos_token}"

dataset = Dataset.from_list([{'text': fmt(d)} for d in train_data])
print(f'✅ {len(dataset)} training examples ready')
for d in train_data[:3]:
    print(f"  • {d['instruction']}")

In [ ]:
# Cell 5: TRAIN! (Takes ~1.5 hours on free T4 GPU)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=100,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir="outputs",
        report_to="none",
        push_to_hub=False,
    ),
)

trainer.train()
print("🎉 TRAINING COMPLETE!")


In [ ]:
# Cell 6: Save & Download to your Mac
model.save_pretrained_gguf('lordy-model', tokenizer, quantization_method='q4_k_m')

import os
gguf = [f for f in os.listdir('lordy-model') if f.endswith('.gguf')][0]
print(f'Model saved as: {gguf}')
print(f'Size: {os.path.getsize(os.path.join("lordy-model", gguf)) / 1e9:.2f} GB')
print()
print('⬇️ Click the folder icon on the left → find lordy-model folder')
print('⬇️ Right-click the .gguf file → Download')
print('⬇️ Transfer to your Mac and load into Ollama!')